<a href="https://colab.research.google.com/github/anoduck/suntime/blob/develop/suntime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Suntime

Discovering time based on the position of the sun.

[Copyright &copy; Anoduck, The Anonymous Duck; 2025](https://anoduck.mit-license.org)

## Prepare Drive for Mounting and Saving Results

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
if not os.path.exists("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results"):
  os.mkdir("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results")

# Variables for runtime
batch = False

In [ ]:
import numpy as np


def save_img(cvimg: np.ndarray, label: str) -> bool:
  write_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/results", f"{label}.jpg")
  cv.imwrite(write_path, cvimg)
  return True

## Locate the Sun

### Environment Setup

#### Install Modules

In [ ]:
!pip install -U opencv-python matplotlib pytesseract
!pip install -U git+https://github.com/pingswept/pysolar

#### Import Modules

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import pytesseract
import numpy as np
from google.colab.patches import cv_imshow as colab_show
import datetime
import pytz
import math

### Image Utils for Analemma

In [ ]:
class Image:
  @classmethod
  def stackImages(cls, imgArray, scale, lables=None):
      if lables is None:
          lables = []
      sizeW = imgArray[0][0].shape[1]
      sizeH = imgArray[0][0].shape[0]
      rows = len(imgArray)
      cols = len(imgArray[0])
      rowsAvailable = isinstance(imgArray[0], list)
      width = imgArray[0][0].shape[1]
      height = imgArray[0][0].shape[0]
      if rowsAvailable:
          for x in range(0, rows):
              for y in range(0, cols):
                  imgArray[x][y] = cv.resize(imgArray[x][y], (int(sizeW * scale), int(sizeH * scale)))
                  if len(imgArray[x][y].shape) == 2: imgArray[x][y] = cv.cvtColor(imgArray[x][y], cv.COLOR_GRAY2BGR)
          imageBlank = np.zeros((height, width, 3), np.uint8)
          hor = [imageBlank] * rows
          hor_con = [imageBlank] * rows
          for x in range(0, rows):
              hor[x] = np.hstack(imgArray[x])
              hor_con[x] = np.concatenate(imgArray[x])
          try:
              ver = np.vstack(hor)
              ver_con = np.concatenate(hor)
          except:
              pass
      else:
          for x in range(0, rows):
              imgArray[x] = cv.resize(imgArray[x], (int(sizeW * scale), int(sizeH * scale)))
              if len(imgArray[x].shape) == 2: imgArray[x] = cv.cvtColor(imgArray[x], cv.COLOR_GRAY2BGR)
          hor = np.hstack(imgArray)
          hor_con = np.concatenate(imgArray)
          ver = hor
      if len(lables) != 0:
          eachImgWidth = int(ver.shape[1] / cols)
          eachImgHeight = int(ver.shape[0] / rows)
          for d in range(0, rows):
              for c in range(0, cols):
                  cv.rectangle(ver, (c * eachImgWidth, eachImgHeight * d),
                                (c * eachImgWidth + len(lables[d][c]) * 13 + 27, 30 + eachImgHeight * d),
                                (255, 255, 255), cv.FILLED)
                  cv.putText(ver, lables[d][c], (eachImgWidth * c + 10, eachImgHeight * d + 20),
                              cv.FONT_HERSHEY_COMPLEX, 0.7, (255, 0, 255), 2)
      return ver

  @classmethod
  def warp(cls, dst: np.ndarray, transformation_matrix: list) -> np.ndarray:
      w, h = dst.shape[:2]
      return cv.warpPerspective(dst, transformation_matrix, (w, h))

  @classmethod
  def get_formated_canny(cls, image: np.ndarray) -> np.ndarray:
      """
      Formats an image to gray, blur, and lastly to canny

      :param image: Source image
      :return: Canny image
      """
      img = image.copy()
      gray = Image.cvt_to_gray(img)
      blur = cv.GaussianBlur(gray, (5, 5), 1)
      canny = cv.Canny(blur, 10, 50)
      return canny

  @classmethod
  def size_reduction(cls, canvas: np.ndarray, size_reduction: float) -> np.ndarray:
      """
      | Reduces image size by cutting of a percentage of pixels starting from the image outlines

      :param canvas: Source image
      :param size_reduction: Percentage of pixels that gets cut of
      """
      canvas = Image.cvt_to_gray(canvas)
      h, w = canvas.shape[:2]
      reduce_pixels_h = int(((h / 100) * size_reduction) / 2)
      reduce_pixels_w = int(((w / 100) * size_reduction) / 2)

      x = reduce_pixels_w
      w = w - reduce_pixels_w
      y = reduce_pixels_h
      h = h - reduce_pixels_h
      return canvas[y:h, x:w]

  @classmethod
  def cvt_to_gray(cls, image: np.ndarray) -> np.ndarray:
      image = image.copy()
      if len(image.shape) < 3:
          return image

      channels = image.shape[2]
      match channels:
          case 3:
              try:
                  image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
              except:
                  try:
                      image = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
                  except:
                      try:
                          image = cv.cvtColor(image, cv.COLOR_HSV2BGR)
                          image = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
                      except:
                          print("Image format not supported")
                          assert ValueError
      return image

  @classmethod
  def show(cls, img: np.ndarray, winname="test", destroy=False) -> None:
      cv.imshow(winname, img)
      cv.waitKey(99999999)
      if destroy:
          cv.destroyWindow(winname)

### Analemma

In [ ]:
def analemma(image) -> tuple:
  blur_radius = int(51)
  last_center = (0.0, 0.0)
  # reduce image size by 20px on all sides and auto converts it to gray
  (h, w) = image.shape[:2]
  h2 = h - 20
  w2 = w - 20
  image_copy = cv.resize(image, (w2, h2))
  image = Image.size_reduction(image, 20)
  print(f"Image copy shape: {image_copy.shape} and image shape: {image.shape}")
  # insure blur_radius is odd
  print(f"blur_radius: {blur_radius}")
  if int(blur_radius) % 2:
      pass
  else:
      blur_radius += 1
  # blur the image
  grey = Image.cvt_to_gray(image)
  try:
      blur = cv.GaussianBlur(grey, (int(blur_radius), int(blur_radius)), cv.BORDER_DEFAULT)
  except Exception:
      blur = cv.medianBlur(grey, int(blur_radius))

  # calculate minMax method
  minMaxMethod = image.copy()
  (minVal, maxVal, minLoc, maxLoc) = cv.minMaxLoc(blur)
  minMaxCenter = maxLoc
  # apply the minMax method
  cv.circle(minMaxMethod, maxLoc, int(blur_radius), (0, 0, 0), 2)
  # prepare the image for the robust method
  thresh = cv.threshold(blur, 210, 225, cv.THRESH_BINARY)[1]
  erode = cv.erode(thresh, None, iterations=7)
  dilate = cv.dilate(erode, None, iterations=4)
  canny = Image.get_formated_canny(dilate)
  # calculate robust method
  points = np.argwhere(canny > 0)
  robustCenter, radius = cv.minEnclosingCircle(points)
  # apply robust method
  robustMethod = image.copy()
  x = int(robustCenter[1])
  y = int(robustCenter[0])
  rad = int(radius)
  cv.circle(robustMethod, (x, y), rad, (300, 100, 100), 2)
  # debug print
  print("lastCenter: " + str(last_center))
  print("dist: " + str(math.dist(robustCenter, last_center)))
  print("robustCenter: " + str(robustCenter))
  print("maxLoc: " + str(robustCenter))
  print("----------------------------------")
  # determine which method to use
  center = minMaxCenter
  if robustCenter == (0.0, 0.0):
      if last_center != (0.0, 0.0):
          if not (math.dist(minMaxCenter, last_center) < 50):
              center = robustCenter
      else:
          center = (0.0, 0.0)
  # put text and highlight the center
  Cx, Cy = center
  # cv2.circle(src, center=(100, 100), radius=50, color=(0, 255, 0), thickness=2)
  image1 = cv.circle(img=image_copy, center=(Cx, Cy), radius=10, color=(31, 95, 255), thickness=-1)
  image2 = cv.putText(
      image1,
      "centroid",
      (Cx - 25, Cy - 25),
      cv.FONT_HERSHEY_SIMPLEX,
      2,
      (31, 95, 255),
      2,
  )
  print(f"Center: {center}, Radius: {int(radius)}")
  colab_show(image2)
  return image2, center

### Perform Calculations

In [ ]:
img = cv.imread('/content/drive/MyDrive/Colab Notebooks/suntime-opencv/IMAG0692_V1xF8JAe.jpg', 1)
sunid_img, center = analemma(img)

## Identify Shadows

### Setup Environment

In [ ]:
import os
import sys

class myenv:

  def __init__(self) -> None:
    self.setup_os()
    self.init_env()
    # self.install_sam2()
    # self.acquire_adaptershadow()
    self.load_imports()
    self.read_image()

  def setup_os(self):
    import os
    os.chdir("/content")
    global CODE_DIR
    CODE_DIR = "/content/suntime"
    print("Done...")

   def init_env(self):
    # !conda info --envs
    # Updating the environment.
    !pip install -U opencv-python matplotlib pytesseract shadowfinder pillow
    !pip install torch torchvision transformers optimum[exporters,onnxruntime] opencv-python pillow datasets samexporter
    # Redundant installations removed for clarity and efficiency
    # !pip install git+https://github.com/IDEA-Research/GroundingDINO.git
    # !pip install git+https://github.com/facebookresearch/segment-anything.git
    print("The environemnt has beed updated...")

  def load_imports(self):
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    import cv2 as cv
    import torch
    from PIL import Image as PIMAGE # Changed import alias to avoid conflict
    # from optimum.onnxruntime import ORTModelForImageSegmentation
    # from transformers import SamProcessor, SamModel # Added SamModel import
    from transformers import pipeline, AutoProcessor, SamProcessor, SamModel
    from datasets import load_dataset
    import tempfile
    import os
    import random as rng
    import numpy as np
    from matplotlib import pyplot as plt
    import google.colab.patches as colab
    from google.colab.patches import cv_imshow as colab_show
    import math
    import glob
    from scipy import ndimage
    import mpl_toolkits.mplot3d.axes3d as p3
    import sys
    import datetime
    import pytz
    import pytesseract
    import shadowfinder
    import onnxruntime as ort # Added onnxruntime import

    print("Done importing modules.")

  def read_image(self):
    import cv2 as cv
    global img # Ensure img is globally accessible
    img = cv.imread('/content/drive/MyDrive/Colab Notebooks/suntime-opencv/IMAG0692_V1xF8JAe.jpg', 1)

In [ ]:
def env_reload():
  gcenv = myenv() # Simply instantiate the class

env_reload()

In [ ]:
# Install required packages
!pip install -U webdataset huggingface_hub transformers scikit-learn
# !pip install "optimum-onnx[onnxruntime-gpu]"@git+https://github.com/huggingface/optimum-onnx.git
!pip install git+https://github.com/facebookresearch/segment-anything-2.git

In [ ]:
test_image = '/content/drive/MyDrive/Colab Notebooks/suntime-opencv/IMAG0692_V1xF8JAe.jpg'  # Replace
processor_dir = "/content/drive/MyDrive/ONNX_Model"
model_id = "facebook/sam2-hiera-large"

### Implementation #7

In [ ]:
import webdataset as wds
from datasets import load_dataset
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
from PIL import Image
import io
import cv2
import os
import logging
from sklearn.metrics import jaccard_score
from transformers import Sam2Processor, Sam2Model, OwlViTProcessor, OwlViTForObjectDetection  # Updated imports
import onnxruntime as ort
from optimum.onnxruntime import ORTOptimizer, ORTQuantizer
from optimum.onnxruntime.configuration import OptimizationConfig, AutoQuantizationConfig
import warnings
warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Step 1: Export, Optimize, and Quantize SAM 2 to ONNX
def export_optimize_quantize_sam(model_id="facebook/sam2-hiera-large", onnx_path="sam2_hiera_large.onnx", optimized_path="sam2_optimized.onnx", quantized_path="sam2_quantized.onnx", processor_dir="sam2_processor"):
    if not os.path.exists(onnx_path):
        # Load SAM 2 model and processor
        model = Sam2Model.from_pretrained(model_id)
        processor = Sam2Processor.from_pretrained(model_id)
        model.eval()

        # Prepare dummy inputs (batch_size=1, num_boxes=1)
        dummy_image = Image.new('RGB', (256, 256))
        inputs = processor(images=dummy_image, input_boxes=[[[0, 0, 100, 100]]], return_tensors="pt")
        pixel_values = inputs["pixel_values"]
        input_boxes = inputs["input_boxes"]  # Shape: [1, 1, 4]

        # Export to ONNX (use opset 17 for Hiera compatibility)
        torch.onnx.export(
            model,
            (pixel_values, input_boxes),
            onnx_path,
            opset_version=17,  # Updated for SAM 2
            input_names=["pixel_values", "input_boxes"],
            output_names=["pred_masks", "iou_scores"],
            dynamic_axes={
                "pixel_values": {0: "batch_size", 2: "height", 3: "width"},
                "input_boxes": {0: "batch_size", 1: "num_boxes"},
                "pred_masks": {0: "batch_size", 1: "num_boxes"}
            }
        )
        logging.info(f"SAM 2 exported to {onnx_path}")

        # Save processor
        processor.save_pretrained(processor_dir)

    # Optimize with ORTOptimizer
    if not os.path.exists(optimized_path):
        optimizer = ORTOptimizer.from_pretrained(os.path.dirname(onnx_path))
        optimization_config = OptimizationConfig(
            optimization_level=2,
            enable_transformers_specific_optimizations=True,
            optimize_for_gpu=torch.cuda.is_available()
        )
        optimizer.optimize(save_dir=os.path.dirname(optimized_path), optimization_config=optimization_config)
        os.rename(os.path.join(os.path.dirname(optimized_path), "model_optimized.onnx"), optimized_path)
        logging.info(f"SAM 2 optimized to {optimized_path}")

    # Quantize with ORTQuantizer
    if not os.path.exists(quantized_path):
        quantizer = ORTQuantizer.from_pretrained(os.path.dirname(optimized_path))
        dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
        quantizer.quantize(save_dir=os.path.dirname(quantized_path), quantization_config=dqconfig)
        os.rename(os.path.join(os.path.dirname(quantized_path), "model_quantized.onnx"), quantized_path)
        logging.info(f"SAM 2 quantized to {quantized_path}")

# Export, optimize, and quantize model (run once)
export_optimize_quantize_sam()

# Step 2: Load Models
sam2_processor = Sam2Processor.from_pretrained("facebook/sam2-hiera-large")  # Updated
sam2_model = Sam2Model.from_pretrained("facebook/sam2-hiera-large")  # Updated

# Load OWLv2 for shadow detection (unchanged)
owl_model_id = "google/owlv2-base-patch16"
owl_processor = OwlViTProcessor.from_pretrained(owl_model_id)
owl_model = OwlViTForObjectDetection.from_pretrained(owl_model_id)

# Step 3: Preprocessing for Edge Cases (unchanged)
def preprocess_image_for_shadows(img):
    if isinstance(img, np.ndarray):
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    img_enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    gamma = 1.2
    look_up_table = np.empty((1,256), np.uint8)
    for i in range(256):
        look_up_table[0,i] = np.clip(pow(i / 255.0, gamma) * 255.0, 0, 255)
    img_gamma = cv2.LUT(img_enhanced, look_up_table)
    return img_gamma

# Step 4: Fallback Shadow Detection (unchanged)
def fallback_shadow_detection(img, threshold=0.3):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    lower_dark = np.array([0, 0, 0])
    upper_dark = np.array([180, 255, int(255 * threshold)])
    dark_mask = cv2.inRange(hsv, lower_dark, upper_dark)
    contours, _ = cv2.findContours(dark_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 10 and h > 10:
            boxes.append([x, y, x+w, y+h])
    return np.array(boxes) if boxes else np.empty((0, 4))

# Step 5: WebDataset Preprocessing (updated for SAM 2 processor)
def preprocess(sample):
    try:
        img = sample.get('image')
        mask = sample.get('shadow_mask')

        # Handle image and mask (unchanged)
        if isinstance(img, bytes):
            img = Image.open(io.BytesIO(img)).convert('RGB')
        elif isinstance(img, np.ndarray):
            img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        elif not isinstance(img, Image.Image):
            logging.warning(f"Unexpected image type: {type(img)}")
            return None

        if isinstance(mask, bytes):
            mask = Image.open(io.BytesIO(mask)).convert('L')
        elif isinstance(mask, np.ndarray):
            mask = Image.fromarray(mask).convert('L')
        elif not isinstance(mask, Image.Image):
            logging.warning(f"Unexpected mask type: {type(mask)}")
            return None

        # Preprocess image (unchanged)
        img_np = np.array(img)
        img_enhanced = preprocess_image_for_shadows(img_np)
        img = Image.fromarray(img_enhanced)

        # OWLv2 for boxes (unchanged)
        owl_inputs = owl_processor(text=["a shadow", "shaded area"], images=img, return_tensors="pt")
        owl_outputs = owl_model(**owl_inputs)
        target_sizes = torch.Tensor([img.size[::-1]])
        results = owl_processor.post_process_object_detection(outputs=owl_outputs, target_sizes=target_sizes, threshold=0.1)
        boxes = results[0]["boxes"].cpu().numpy()

        if len(boxes) == 0:
            boxes = fallback_shadow_detection(img_enhanced)
            if len(boxes) == 0:
                logging.warning("No shadows detected.")
                return None

        # Ensure boxes are in correct shape: [1, num_boxes, 4] (unchanged)
        boxes = boxes.reshape(1, -1, 4)

        # Transforms (unchanged)
        transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        mask_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

        img_tensor = transform(img)
        mask_tensor = mask_transform(mask)
        mask_tensor = (mask_tensor > 0.5).float()

        return {"jpg": img_tensor, "mask": mask_tensor, "boxes": boxes}
    except Exception as e:
        logging.error(f"Error processing sample: {e}")
        return None

# Step 6: Create WebDataset (unchanged)
def create_webdataset_from_hf(dataset_name="emasquil/shadow-eo", split="train"):
    try:
        dataset = load_dataset(dataset_name, split=split, streaming=True)
        wds_dataset = wds.WebDataset(dataset).map(preprocess).filter(lambda x: x is not None).shuffle(1000)
        return wds_dataset
    except Exception as e:
        logging.error(f"Failed to load dataset: {e}")
        raise

# Step 7: Configure DataLoader (unchanged)
def get_dataloader(wds_dataset, batch_size=4, num_workers=2):
    def collate_fn(batch):
        boxes = [x["boxes"] for x in batch]
        max_boxes = max(b.shape[1] for b in boxes)
        padded_boxes = []
        for b in boxes:
            if b.shape[1] < max_boxes:
                pad = np.zeros((1, max_boxes - b.shape[1], 4), dtype=np.float32)
                b = np.concatenate([b, pad], axis=1)
            padded_boxes.append(b)
        return {
            "jpg": torch.stack([x["jpg"] for x in batch]),
            "mask": torch.stack([x["mask"] for x in batch]),
            "boxes": torch.tensor(np.concatenate(padded_boxes, axis=0))
        }

    dataloader = DataLoader(
        wds_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False
    )
    return dataloader

# Step 8: Training Loop (updated for SAM 2)
def train_sam(model, dataloader, num_epochs=5, learning_rate=5e-6, device="cuda" if torch.cuda.is_available() else "cpu"):  # Lower LR for SAM 2
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, batch in enumerate(dataloader):
            images = batch["jpg"].to(device)
            masks = batch["mask"].to(device)
            boxes = batch["boxes"].to(device)

            # Convert images back to PIL for processor
            pil_images = [Image.fromarray((img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)) for img in images]
            inputs = sam2_processor(images=pil_images, input_boxes=boxes, return_tensors="pt")  # Updated processor
            inputs = {k: v.to(device) for k, v in inputs.items()}

            optimizer.zero_grad()
            outputs = model(**inputs)
            pred_masks = outputs.pred_masks
            loss = criterion(pred_masks, masks)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            if i % 10 == 0:
                logging.info(f"Epoch {epoch+1}/{num_epochs}, Batch {i}, Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(dataloader.dataset)
        logging.info(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {epoch_loss:.4f}")

    return model

# Step 9: Inference with Quantized ONNX (updated for SAM 2)
def infer_with_onnx(image_path, onnx_path="sam2_quantized.onnx", processor_dir="sam2_processor"):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Invalid image path")

    image_enhanced = preprocess_image_for_shadows(image)
    image_pil = Image.fromarray(cv2.cvtColor(image_enhanced, cv2.COLOR_RGB2BGR))

    boxes = detect_shadows_owlvit(image_pil)
    if len(boxes) == 0:
        boxes = fallback_shadow_detection(image_enhanced)

    if len(boxes) == 0:
        logging.info("No shadows detected.")
        return None

    processor = SamProcessor.from_pretrained(processor_dir)  # Updated
    boxes = boxes.reshape(1, -1, 4)
    inputs = processor(image_pil, input_boxes=[boxes.tolist()], return_tensors="pt")

    # Load quantized ONNX model
    session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider" if torch.cuda.is_available() else "CPUExecutionProvider"])
    pixel_values = inputs["pixel_values"].numpy()
    input_boxes = inputs["input_boxes"].numpy()
    outputs = session.run(None, {"pixel_values": pixel_values, "input_boxes": input_boxes})
    pred_masks = outputs[0]

    masks = processor.post_process_masks(  # Updated post-processing
        torch.from_numpy(pred_masks),
        inputs["original_sizes"],
        inputs["reshaped_input_sizes"]
    )[0].squeeze(1).numpy()

    original_image = image.copy()
    for mask in masks:
        kernel = np.ones((3,3), np.uint8)
        mask_refined = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        color = np.array([0, 255, 0], dtype=np.uint8)
        original_image[mask_refined > 0] = original_image[mask_refined > 0] * 0.5 + color * 0.5

    output_path = "shadow_segmented_image.jpg"
    cv2.imwrite(output_path, original_image)
    logging.info(f"Segmented image saved as {output_path}")
    return masks

# Step 10: OWLv2 Shadow Detection (unchanged)
def detect_shadows_owlvit(image_pil, text_prompts=["a shadow", "shaded area"]):
    inputs = owl_processor(text=text_prompts, images=image_pil, return_tensors="pt")
    outputs = owl_model(**inputs)
    target_sizes = torch.Tensor([image_pil.size[::-1]])
    results = owl_processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=0.1)
    boxes = results[0]["boxes"].cpu().numpy()
    return boxes

# Step 11: Main Execution (updated variable names)
if __name__ == "__main__":
    # Project settings for suntime
    dataset_name = "emasquil/shadow-eo"
    batch_size = 4
    num_epochs = 5

    # Create dataset and dataloader
    wds_dataset = create_webdataset_from_hf(dataset_name=dataset_name, split="train")
    dataloader = get_dataloader(wds_dataset, batch_size=batch_size, num_workers=2)

    # Train SAM 2
    trained_model = train_sam(sam2_model, dataloader, num_epochs=num_epochs)

    # Save model
    save_path = "shadow_sam2_model.pth"
    torch.save(trained_model.state_dict(), save_path)
    logging.info(f"Model saved to {save_path}")

    # Test inference with quantized ONNX
    # test_image = "path/to/test_image.jpg"  # Replace
    masks = infer_with_onnx(test_image)

    # Evaluate on test split
    test_dataset = create_webdataset_from_hf(dataset_name=dataset_name, split="test")
    test_dataloader = get_dataloader(test_dataset, batch_size=1, num_workers=1)
    trained_model.eval()
    with torch.no_grad():
        for batch in test_dataloader:
            img = batch["jpg"].to("cuda" if torch.cuda.is_available() else "cpu")
            mask = batch["mask"].cpu().numpy()
            boxes = batch["boxes"]
            inputs = sam2_processor(  # Updated
                images=[Image.fromarray((img[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8))],
                input_boxes=boxes[0:1].tolist(),
                return_tensors="pt"
            )
            inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
            pred = trained_model(**inputs).pred_masks.cpu().numpy()
            pred_binary = (pred > 0.5).astype(np.uint8)
            iou = jaccard_score(mask.flatten(), pred_binary.flatten())
            logging.info(f"Test IoU: {iou:.4f}")
            break

### Implementation #6

In [ ]:
model_save_path = "/content/drive/MyDrive/ONNX_Model"
import webdataset as wds
from datasets import load_dataset
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
from PIL import Image
import io
import cv2
import os
import logging
from sklearn.metrics import jaccard_score
from transformers import AutoProcessor, AutoModelForMaskGeneration, OwlViTProcessor, OwlViTForObjectDetection
import onnxruntime as ort
from optimum.onnxruntime import ORTOptimizer, ORTQuantizer
from optimum.onnxruntime.configuration import OptimizationConfig, AutoQuantizationConfig
import warnings
warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Step 1: Export, Optimize, and Quantize SAM to ONNX
def export_optimize_quantize_sam(model_id="facebook/sam-vit-large", onnx_path="sam_vit_large.onnx", optimized_path="sam_optimized.onnx", quantized_path="sam_quantized.onnx", processor_dir="sam_processor"):
    if not os.path.exists(onnx_path):
        # Load SAM model and processor
        model = AutoModelForMaskGeneration.from_pretrained(model_id)
        processor = AutoProcessor.from_pretrained(model_id)
        model.eval()

        # Prepare dummy inputs (batch_size=1, num_boxes=1)
        dummy_image = Image.new('RGB', (256, 256))
        inputs = processor(images=dummy_image, input_boxes=[[[0, 0, 100, 100]]], return_tensors="pt")
        pixel_values = inputs["pixel_values"]
        input_boxes = inputs["input_boxes"]  # Shape: [1, 1, 4]

        # Export to ONNX
        torch.onnx.export(
            model,
            (pixel_values, input_boxes),
            onnx_path,
            opset_version=14,
            input_names=["pixel_values", "input_boxes"],
            output_names=["pred_masks", "iou_scores"],
            dynamic_axes={
                "pixel_values": {0: "batch_size", 2: "height", 3: "width"},
                "input_boxes": {0: "batch_size", 1: "num_boxes"},
                "pred_masks": {0: "batch_size", 1: "num_boxes"}
            }
        )
        logging.info(f"SAM exported to {onnx_path}")

        # Save processor
        processor.save_pretrained(processor_dir)

    # Optimize with ORTOptimizer
    if not os.path.exists(optimized_path):
        optimizer = ORTOptimizer.from_pretrained(os.path.dirname(onnx_path))
        optimization_config = OptimizationConfig(
            optimization_level=2,
            enable_transformers_specific_optimizations=True,
            optimize_for_gpu=torch.cuda.is_available()
        )
        optimizer.optimize(save_dir=os.path.dirname(optimized_path), optimization_config=optimization_config)
        os.rename(os.path.join(os.path.dirname(optimized_path), "model_optimized.onnx"), optimized_path)
        logging.info(f"SAM optimized to {optimized_path}")

    # Quantize with ORTQuantizer
    if not os.path.exists(quantized_path):
        quantizer = ORTQuantizer.from_pretrained(os.path.dirname(optimized_path))
        dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
        quantizer.quantize(save_dir=os.path.dirname(quantized_path), quantization_config=dqconfig)
        os.rename(os.path.join(os.path.dirname(quantized_path), "model_quantized.onnx"), quantized_path)
        logging.info(f"SAM quantized to {quantized_path}")

# Export, optimize, and quantize model (run once)
export_optimize_quantize_sam()

# Step 2: Load Models
sam_processor = AutoProcessor.from_pretrained("facebook/sam-vit-large")
sam_model = AutoModelForMaskGeneration.from_pretrained("facebook/sam-vit-large")

# Load OWLv2 for shadow detection
owl_model_id = "google/owlv2-base-patch16"
owl_processor = OwlViTProcessor.from_pretrained(owl_model_id)
owl_model = OwlViTForObjectDetection.from_pretrained(owl_model_id)

# Step 3: Preprocessing for Edge Cases
def preprocess_image_for_shadows(img):
    if isinstance(img, np.ndarray):
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    img_enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    gamma = 1.2
    look_up_table = np.empty((1,256), np.uint8)
    for i in range(256):
        look_up_table[0,i] = np.clip(pow(i / 255.0, gamma) * 255.0, 0, 255)
    img_gamma = cv2.LUT(img_enhanced, look_up_table)
    return img_gamma

# Step 4: Fallback Shadow Detection
def fallback_shadow_detection(img, threshold=0.3):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    lower_dark = np.array([0, 0, 0])
    upper_dark = np.array([180, 255, int(255 * threshold)])
    dark_mask = cv2.inRange(hsv, lower_dark, upper_dark)
    contours, _ = cv2.findContours(dark_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 10 and h > 10:
            boxes.append([x, y, x+w, y+h])
    return np.array(boxes) if boxes else np.empty((0, 4))

# Step 5: WebDataset Preprocessing
def preprocess(sample):
    try:
        img = sample.get('image')
        mask = sample.get('shadow_mask')

        # Handle image
        if isinstance(img, bytes):
            img = Image.open(io.BytesIO(img)).convert('RGB')
        elif isinstance(img, np.ndarray):
            img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        elif not isinstance(img, Image.Image):
            logging.warning(f"Unexpected image type: {type(img)}")
            return None

        # Handle mask
        if isinstance(mask, bytes):
            mask = Image.open(io.BytesIO(mask)).convert('L')
        elif isinstance(mask, np.ndarray):
            mask = Image.fromarray(mask).convert('L')
        elif not isinstance(mask, Image.Image):
            logging.warning(f"Unexpected mask type: {type(mask)}")
            return None

        # Preprocess image
        img_np = np.array(img)
        img_enhanced = preprocess_image_for_shadows(img_np)
        img = Image.fromarray(img_enhanced)

        # OWLv2 for boxes
        owl_inputs = owl_processor(text=["a shadow", "shaded area"], images=img, return_tensors="pt")
        owl_outputs = owl_model(**owl_inputs)
        target_sizes = torch.Tensor([img.size[::-1]])
        results = owl_processor.post_process_object_detection(outputs=owl_outputs, target_sizes=target_sizes, threshold=0.1)
        boxes = results[0]["boxes"].cpu().numpy()

        if len(boxes) == 0:
            boxes = fallback_shadow_detection(img_enhanced)
            if len(boxes) == 0:
                logging.warning("No shadows detected.")
                return None

        # Ensure boxes are in correct shape: [1, num_boxes, 4]
        boxes = boxes.reshape(1, -1, 4)

        # Transforms
        transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        mask_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])

        img_tensor = transform(img)
        mask_tensor = mask_transform(mask)
        mask_tensor = (mask_tensor > 0.5).float()

        return {"jpg": img_tensor, "mask": mask_tensor, "boxes": boxes}
    except Exception as e:
        logging.error(f"Error processing sample: {e}")
        return None

# Step 6: Create WebDataset
def create_webdataset_from_hf(dataset_name="emasquil/shadow-eo", split="train"):
    try:
        dataset = load_dataset(dataset_name, split=split, streaming=True)
        wds_dataset = wds.WebDataset(dataset).map(preprocess).filter(lambda x: x is not None).shuffle(1000)
        return wds_dataset
    except Exception as e:
        logging.error(f"Failed to load dataset: {e}")
        raise

# Step 7: Configure DataLoader
def get_dataloader(wds_dataset, batch_size=4, num_workers=2):
    def collate_fn(batch):
        # Ensure boxes are properly stacked as [batch_size, num_boxes, 4]
        boxes = [x["boxes"] for x in batch]
        max_boxes = max(b.shape[1] for b in boxes)
        padded_boxes = []
        for b in boxes:
            if b.shape[1] < max_boxes:
                # Pad with zeros if fewer boxes
                pad = np.zeros((1, max_boxes - b.shape[1], 4), dtype=np.float32)
                b = np.concatenate([b, pad], axis=1)
            padded_boxes.append(b)
        return {
            "jpg": torch.stack([x["jpg"] for x in batch]),
            "mask": torch.stack([x["mask"] for x in batch]),
            "boxes": torch.tensor(np.concatenate(padded_boxes, axis=0))
        }

    dataloader = DataLoader(
        wds_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False
    )
    return dataloader

# Step 8: Training Loop
def train_sam(model, dataloader, num_epochs=5, learning_rate=1e-5, device="cuda" if torch.cuda.is_available() else "cpu"):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, batch in enumerate(dataloader):
            images = batch["jpg"].to(device)
            masks = batch["mask"].to(device)
            boxes = batch["boxes"].to(device)  # Shape: [batch_size, num_boxes, 4]

            # Convert images back to PIL for processor
            pil_images = [Image.fromarray((img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)) for img in images]
            inputs = sam_processor(images=pil_images, input_boxes=boxes, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}

            optimizer.zero_grad()
            outputs = model(**inputs)
            pred_masks = outputs.pred_masks
            loss = criterion(pred_masks, masks)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            if i % 10 == 0:
                logging.info(f"Epoch {epoch+1}/{num_epochs}, Batch {i}, Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(dataloader.dataset)
        logging.info(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {epoch_loss:.4f}")

    return model

# Step 9: Inference with Quantized ONNX
def infer_with_onnx(image_path, onnx_path="sam_quantized.onnx", processor_dir="sam_processor"):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Invalid image path")

    image_enhanced = preprocess_image_for_shadows(image)
    image_pil = Image.fromarray(cv2.cvtColor(image_enhanced, cv2.COLOR_RGB2BGR))

    boxes = detect_shadows_owlvit(image_pil)
    if len(boxes) == 0:
        boxes = fallback_shadow_detection(image_enhanced)

    if len(boxes) == 0:
        logging.info("No shadows detected.")
        return None

    processor = AutoProcessor.from_pretrained(processor_dir)
    # Ensure boxes are [1, num_boxes, 4]
    boxes = boxes.reshape(1, -1, 4)
    inputs = processor(image_pil, input_boxes=[boxes.tolist()], return_tensors="pt")

    # Load quantized ONNX model
    session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider" if torch.cuda.is_available() else "CPUExecutionProvider"])
    pixel_values = inputs["pixel_values"].numpy()
    input_boxes = inputs["input_boxes"].numpy()
    outputs = session.run(None, {"pixel_values": pixel_values, "input_boxes": input_boxes})
    pred_masks = outputs[0]

    masks = processor.post_process_masks(
        torch.from_numpy(pred_masks),
        inputs["original_sizes"],
        inputs["reshaped_input_sizes"]
    )[0].squeeze(1).numpy()

    original_image = image.copy()
    for mask in masks:
        kernel = np.ones((3,3), np.uint8)
        mask_refined = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        color = np.array([0, 255, 0], dtype=np.uint8)
        original_image[mask_refined > 0] = original_image[mask_refined > 0] * 0.5 + color * 0.5

    output_path = "shadow_segmented_image.jpg"
    cv2.imwrite(output_path, original_image)
    logging.info(f"Segmented image saved as {output_path}")
    return masks

# Step 10: OWLv2 Shadow Detection
def detect_shadows_owlvit(image_pil, text_prompts=["a shadow", "shaded area"]):
    inputs = owl_processor(text=text_prompts, images=image_pil, return_tensors="pt")
    outputs = owl_model(**inputs)
    target_sizes = torch.Tensor([image_pil.size[::-1]])
    results = owl_processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=0.1)
    boxes = results[0]["boxes"].cpu().numpy()
    return boxes

# Step 11: Main Execution
if __name__ == "__main__":
    # Project settings for suntime
    dataset_name = "emasquil/shadow-eo"
    batch_size = 4
    num_epochs = 5

    # Create dataset and dataloader
    wds_dataset = create_webdataset_from_hf(dataset_name=dataset_name, split="train")
    dataloader = get_dataloader(wds_dataset, batch_size=batch_size, num_workers=2)

    # Train SAM
    trained_model = train_sam(sam_model, dataloader, num_epochs=num_epochs)

    # Save model
    save_path = "shadow_sam_model.pth"
    torch.save(trained_model.state_dict(), save_path)
    logging.info(f"Model saved to {save_path}")

    # Test inference with quantized ONNX
    test_image = '/content/drive/MyDrive/Colab Notebooks/suntime-opencv/IMAG0692_V1xF8JAe.jpg'  # Replace
    masks = infer_with_onnx(test_image)

    # Evaluate on test split
    test_dataset = create_webdataset_from_hf(dataset_name=dataset_name, split="test")
    test_dataloader = get_dataloader(test_dataset, batch_size=1, num_workers=1)
    trained_model.eval()
    with torch.no_grad():
        for batch in test_dataloader:
            img = batch["jpg"].to("cuda" if torch.cuda.is_available() else "cpu")
            mask = batch["mask"].cpu().numpy()
            boxes = batch["boxes"]
            inputs = sam_processor(
                images=[Image.fromarray((img[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8))],
                input_boxes=boxes[0:1].tolist(),  # Ensure [1, num_boxes, 4]
                return_tensors="pt"
            )
            inputs = {k: v.to("cuda" if torch.cuda.is_available() else "cpu") for k, v in inputs.items()}
            pred = trained_model(**inputs).pred_masks.cpu().numpy()
            pred_binary = (pred > 0.5).astype(np.uint8)
            iou = jaccard_score(mask.flatten(), pred_binary.flatten())
            logging.info(f"Test IoU: {iou:.4f}")
            break

### Implementation #5

In [ ]:
import webdataset as wds
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForMaskGeneration
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np
from PIL import Image as Pimage
from sklearn.metrics import jaccard_score
import cv2
import io
import os
from huggingface_hub import HfApi
import logging

# Configure logging for debugging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


dataset_name = "emasquil/shadow-eo"
image_size = (256, 256)
batch_size = 8
num_epochs = 5


# Step 1: Define the Model (Simple CNN for Shadow Segmentation)
# class ShadowSegmentationCNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
#         self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
#         self.conv3 = nn.Conv2d(32, 1, kernel_size=3, padding=1)
#         self.relu = nn.ReLU()
#         self.pool = nn.MaxPool2d(2, 2)
#         self.upsample = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)

#     def forward(self, x):
#         x = self.relu(self.conv1(x))
#         x = self.pool(x)
#         x = self.relu(self.conv2(x))
#         x = self.pool(x)
#         x = self.conv3(x)
#         x = self.upsample(x)
#         return torch.sigmoid(x)

# Step 2: Prepare WebDataset from Hugging Face Hub
def create_webdataset_from_hf(dataset_name="emasquil/shadow-eo", split="train", image_size=(256, 256)):
    """
    Load a Hugging Face dataset and convert to WebDataset format for streaming.
    Assumes dataset has 'image' (PIL) and 'shadow_mask' (binary mask).
    """
    # Load dataset from Hugging Face Hub with streaming
    dataset = load_dataset(dataset_name, split=split, streaming=True)

    # Transform function for images and masks
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def preprocess(sample):
        try:
            img = sample['image']
            mask = sample['shadow_mask']
            print(f'Contents of sample: {sample}')
            print(f'Type of sample: {type(sample)}')
            print(f'Image is type: {type(sample["image"])}')
            print(f'Mask is type: {type(sample["shadow_mask"])}')

            # Convert to PIL if not already
            if not isinstance(img, Pimage.Image):
                img = Pimage.fromarray(img)
            if not isinstance(mask, Pimage.Image):
                mask = Pimage.fromarray(mask)

            inputs = processor(img, return_tensors="pt")
            inputs['labels'] = processor(mask, return_tensors="pt")['pixel_values']
            return inputs

            # Apply transforms
            # img = transform(img)
            # mask = transforms.Resize(image_size)(mask)
            # mask = transforms.ToTensor()(mask)
            # mask = (mask > 0.5).float()  # Binarize mask

            # return {"jpg": img, "mask": mask}
        except Exception as e:
            logging.warning(f"Error processing sample: {e}")
            return None

    # def preprocess(examples):
    #     # Assuming the dataset has 'image' and 'mask' columns based on the dataset card
    #     inputs = processor(examples['image'], return_tensors="pt")
    #     inputs['labels'] = processor(examples['mask'], return_tensors="pt")['pixel_values']
    #     return inputs

    # Create WebDataset
    # Since datasets streaming doesn't directly produce tar files, we simulate WebDataset's iterator
    wds_dataset = wds.WebDataset(dataset, shardshuffle=3).map(preprocess).filter(lambda x: x is not None)

    return wds_dataset

# Step 3: Configure PyTorch DataLoader
def get_dataloader(wds_dataset, batch_size=8, num_workers=2):
    """
    Wrap WebDataset in a PyTorch DataLoader.
    """
    # Configure DataLoader with WebDataset
    dataloader = DataLoader(
        wds_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        collate_fn=lambda batch: {
            "jpg": torch.stack([x["jpg"] for x in batch]),
            "mask": torch.stack([x["mask"] for x in batch])
        },
        pin_memory=True if torch.cuda.is_available() else False
    )
    return dataloader

# Step 4: Training Loop
def train_model(model, dataloader, num_epochs=5, learning_rate=1e-3, device="cuda" if torch.cuda.is_available() else "cpu"):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()  # Binary cross-entropy for segmentation

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for batch in dataloader:
            images = batch["jpg"].to(device)
            masks = batch["mask"].to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(dataloader.dataset)
        logging.info(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

    return model

# Step 5: Main Execution
if __name__ == "__main__":
    dataset = load_dataset("emasquil/shadow-eo", split="train", streaming=True)
    for sample in dataset.take(1):
      print(sample.keys())
      print(type(sample['tif']))
      print(sample['__key__'])
      print(type(sample['tif']), type(sample['shadow_mask']))
    # Initialize WebDataset
    wds_dataset = create_webdataset_from_hf(dataset_name=dataset_name, split="train", image_size=image_size)

    # Create DataLoader
    dataloader = get_dataloader(wds_dataset, batch_size=batch_size, num_workers=2)

    # Initialize and train model
    # model = ShadowSegmentationCNN()
    processor = AutoProcessor.from_pretrained("facebook/sam-vit-large")
    model = AutoModelForMaskGeneration.from_pretrained("facebook/sam-vit-large")
    trained_model = train_model(model, dataloader, num_epochs=num_epochs)

    # Save model
    save_path = "shadow_segmentation_model.pth"
    torch.save(trained_model.state_dict(), save_path)
    logging.info(f"Model saved to {save_path}")

    # Optional: Evaluate on a test sample
    test_dataset = create_webdataset_from_hf(dataset_name=dataset_name, split="test", image_size=image_size)
    test_dataloader = get_dataloader(test_dataset, batch_size=1, num_workers=1)
    model.eval()
    with torch.no_grad():
        for batch in test_dataloader:
            img = batch["jpg"].to("cuda" if torch.cuda.is_available() else "cpu")
            mask = batch["mask"].cpu().numpy()
            pred = model(img).cpu().numpy()
            pred_binary = (pred > 0.5).astype(np.uint8)
            iou = jaccard_score(mask.flatten(), pred_binary.flatten())
            logging.info(f"Test IoU: {iou:.4f}")
            break  # One sample for demo

### Hugging Face implementation

In [ ]:
!pip install webdataset
!pip install huggingface_hub

import webdataset as wds
from huggingface_hub import HfFileSystem, get_token, hf_hub_url
from torch.utils.data import DataLoader

# Login using e.g. `huggingface-cli login` to access this dataset
fs = HfFileSystem()
files = [fs.resolve_path(path) for path in fs.glob("hf://datasets/emasquil/shadow-eo/**/*")]
urls = [hf_hub_url(file.repo_id, file.path_in_repo, repo_type="dataset") for file in files]
urls = f"pipe: curl -s -L -H 'Authorization:Bearer {get_token()}' {'::'.join(urls)}"

dataset = wds.WebDataset(urls, shardshuffle=3).decode()
dataloader = DataLoader(dataset, batch_size=64, num_workers=2)

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor, Trainer, TrainingArguments
from PIL import Image

processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
# dataset = load_dataset("ParityError/ControlNet-Shadows")
print(dataset)

# image = Image.open("/content/drive/MyDrive/Colab Notebooks/suntime-opencv/IMAG0692_V1xF8JAe.jpg")

# Prepare data (preprocess images and masks – assume binary labels: 0=non-shadow, 1=shadow)
def preprocess(examples):
    # Assuming the dataset has 'image' and 'mask' columns based on the dataset card
    inputs = processor(examples['image'], return_tensors="pt")
    inputs['labels'] = processor(examples['mask'], return_tensors="pt")['pixel_values']
    return inputs

train_dataset = dataset['train'].map(preprocess, batched=True)

# Load model (2 classes: shadow/non-shadow)
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512", num_labels=2)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

# Training setup
training_args = TrainingArguments(
    output_dir="./shadow_detector",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    logging_steps=100,
    remove_unused_columns=False, # Keep columns for feature extraction
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()  # Fine-tune the model
trainer.save_model("./shadow_detector")  # Save for later use

In [ ]:
from transformers import pipeline
import matplotlib.pyplot as plt
from PIL import Image

# Load a pre-trained segmentation pipeline (e.g., SegFormer fine-tuned on ADE20K, which includes scene understanding)
segmentor = pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-ade-512-512")

# Load your image (replace with your file path or URL)
image = Image.open("your_image.jpg")

# Run segmentation
outputs = segmentor(image)

# Display results (shadows might appear in categories like "ground" or "dark areas" – inspect outputs)
for output in outputs:
    print(output['label'])  # Labels like 'sky', 'building', etc. – adapt for shadows
    plt.imshow(output['mask'])
    plt.show()

### Use PyTorch, ONNX, and SAM to detect shadows

In [ ]:
from transformers import SegformerImageProcessor
from PIL import Image
import io
import torch

processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")

# Create lists to store processed data
processed_images = []
processed_labels = []

# Iterate through the WebDataset and preprocess each sample
for sample in dataset:
    # Assuming the dataset yields dictionaries with 'jpg' for image and 'png' for mask
    # You might need to adjust the keys based on your dataset's structure
    if 'jpg' in sample and 'png' in sample:
        try:
            # Open image from bytes
            image = Image.open(io.BytesIO(sample['jpg']))
            # Open mask from bytes (ensure it's in a format PIL can read, like PNG)
            mask = Image.open(io.BytesIO(sample['png'])).convert("L") # Convert mask to grayscale

            # Preprocess image and mask
            inputs = processor(image, return_tensors="pt")
            # Resize and process mask to match the model's expected label size
            labels = processor(mask, return_tensors="pt", size={"height": inputs["pixel_values"].shape[-2], "width": inputs["pixel_values"].shape[-1]})['pixel_values']

            processed_images.append(inputs['pixel_values'])
            processed_labels.append(labels)

        except Exception as e:
            print(f"Error processing sample: {e}")
            # Optionally, skip the sample or handle the error as needed

# You can now convert the processed lists to PyTorch tensors or use them as needed
# Note: This approach might be memory-intensive for very large datasets.
# For larger datasets, consider processing in batches or using a custom data loader with WebDataset.

# Example: Convert to tensors (if all images have the same shape)
if processed_images and processed_labels:
    try:
        processed_images_tensor = torch.cat(processed_images, dim=0)
        processed_labels_tensor = torch.cat(processed_labels, dim=0)
        print("Successfully processed data into tensors.")
        print(f"Processed images shape: {processed_images_tensor.shape}")
        print(f"Processed labels shape: {processed_labels_tensor.shape}")
    except Exception as e:
        print(f"Could not concatenate tensors. Ensure all processed images/labels have the same shape: {e}")